In [1]:
import pandas as pd
df = pd.read_csv("PC_Uncertainty_dataset.csv")

In [2]:
df

,Gross Electrical Output (MWg),As-Delivered Primary Fuel Cost ($/ton),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg),Important Performance & Cost Factors: CO2 Removed (tons/hr),Important Performance & Cost Factors: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)
0,1534.0,48.900002,38.299999,60.877499,3.6775,868.279724,1336.483643,159.369995
1,2122.0,45.500000,35.700001,87.982498,4.8325,1722.348755,1825.054810,110.072769
2,1498.0,67.300003,42.900002,62.437500,1.5075,869.229858,1337.637817,166.645416
3,418.0,31.700001,28.700001,89.542503,4.5175,345.206390,360.549957,139.354156
4,1666.0,57.500000,11.500000,64.387497,1.5775,996.329956,1486.579224,159.967117
...,...,...,...,...,...,...,...,...
195,202.0,58.099998,38.700001,70.432503,2.4175,131.969788,178.349472,223.930115
196,1462.0,61.500000,31.100000,87.592499,5.1475,1181.545776,1252.625000,122.992653
197,646.0,60.900002,48.700001,66.142502,4.4475,396.695221,557.657959,178.050934
198,262.0,37.900002,29.900000,62.827499,7.4925,153.009171,218.053696,219.787796


In [3]:
df.isna().sum()

Gross Electrical Output (MWg)                                       0
As-Delivered Primary Fuel Cost ($/ton)                              0
Cooling Water Temperature Rise (°F)                                 0
Absorber CO2 Removal Efficiency (%)                                 0
Amine Scrubber Power Requirement (% MWg)                            0
Important Performance & Cost Factors: CO2 Removed (tons/hr)         0
Important Performance & Cost Factors: Net Electrical Output (MW)    0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)       0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Gross Electrical Output (MWg)',
    'As-Delivered Primary Fuel Cost ($/ton)',
    'Cooling Water Temperature Rise (°F)',
    'Absorber CO2 Removal Efficiency (%)',
    'Amine Scrubber Power Requirement (% MWg)']]

y = df['Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)']

In [6]:
x

,Gross Electrical Output (MWg),As-Delivered Primary Fuel Cost ($/ton),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg)
0,1534.0,48.900002,38.299999,60.877499,3.6775
1,2122.0,45.500000,35.700001,87.982498,4.8325
2,1498.0,67.300003,42.900002,62.437500,1.5075
3,418.0,31.700001,28.700001,89.542503,4.5175
4,1666.0,57.500000,11.500000,64.387497,1.5775
...,...,...,...,...,...
195,202.0,58.099998,38.700001,70.432503,2.4175
196,1462.0,61.500000,31.100000,87.592499,5.1475
197,646.0,60.900002,48.700001,66.142502,4.4475
198,262.0,37.900002,29.900000,62.827499,7.4925


In [7]:
y

0      159.369995
1      110.072769
2      166.645416
3      139.354156
4      159.967117
          ...    
195    223.930115
196    122.992653
197    178.050934
198    219.787796
199    167.739761
Name: Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== REGRESSION MODELS ===\n")


=== REGRESSION MODELS ===



In [9]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.7870501234431588
Test R2: 0.7743253426875273


In [10]:
# SVR
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])
svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["linear", "rbf"],
        "svr__C": [0.01, 0.1, 1, 10,100],
        "svr__gamma": ["scale"]
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(x_train, y_train)

best_svr = svr_grid.best_estimator_
print("SVR best params:", svr_grid.best_params_)
print('Train R2:',best_svr.score(x_train,y_train))
print('Test R2:',best_svr.score(x_test,y_test))

SVR best params: {'svr__C': 100, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
Train R2: 0.927342381992479
Test R2: 0.8422696688346641


In [11]:
# XGBoost Regressor
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(x_train, y_train)

best_xgb = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)
print('Train R2:',best_xgb.score(x_train,y_train))
print('Test R2:',best_xgb.score(x_test,y_test))

XGBoost best params: {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.9999999752708596
Test R2: 0.952744357208698


In [12]:
# Gradient Boosting Regressor
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "max_features": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(x_train, y_train)

best_gbr = gbr_grid.best_estimator_
print("Gradient Boosting best params:", gbr_grid.best_params_)
print('Train R2:', best_gbr.score(x_train, y_train))
print('Test R2:', best_gbr.score(x_test, y_test))

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 0.8, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.9999817449058069
Test R2: 0.9670396763275443


In [13]:
# Random Forest Regressor
rfr_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100,300],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ['sqrt',0.5, 0.7]
    },
    cv=5, scoring="r2", n_jobs=-1
)
rfr_grid.fit(x_train, y_train)

best_rfr = rfr_grid.best_estimator_
print("RF best params:", rfr_grid.best_params_)
print('Train R2:',best_rfr.score(x_train,y_train))
print('Test R2:',best_rfr.score(x_test,y_test))

RF best params: {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Train R2: 0.9886557177957375
Test R2: 0.9405579398632956


In [14]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
grid = GridSearchCV(poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1)

In [15]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 4, 'reg__alpha': 0.001}
Train R2: 0.9931290340488487
Test R2: 0.8814123394596582


In [16]:
#Exporting to excel
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file       = 'Cost_of_CO2_Captured.xlsx'
experiment = 'Target_3_Cost_of_CO2_Captured'   # change per experiment

results = []

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Experiment': experiment,
        'Model':      name,
        'Train R2':   round(model.score(x_train, y_train), 4),
        'Test R2':    round(model.score(x_test, y_test), 4),
        'MAE':        round(mean_absolute_error(y_test, y_pred), 4),
        'MSE':        round(mean_squared_error(y_test, y_pred), 4),
        'RMSE':       round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE (%)':   round(np.mean(np.abs((y_test - y_pred) / y_test)) * 100, 4)
    })

results_df = pd.DataFrame(results)

if os.path.exists(file):
    existing = pd.read_excel(file)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_excel(file, index=False)
else:
    results_df.to_excel(file, index=False)

print(results_df)

                      Experiment              Model  Train R2  Test R2  \
0  Target_3_Cost_of_CO2_Captured  Linear Regression    0.7871   0.7743   
1  Target_3_Cost_of_CO2_Captured                SVR    0.9273   0.8423   
2  Target_3_Cost_of_CO2_Captured      Random Forest    0.9887   0.9406   
3  Target_3_Cost_of_CO2_Captured            XGBoost    1.0000   0.9527   
4  Target_3_Cost_of_CO2_Captured     Gradient Boost    1.0000   0.9670   
5  Target_3_Cost_of_CO2_Captured         Polynomial    0.9931   0.8814   

       MAE       MSE     RMSE  MAPE (%)  
0  11.5271  256.2444  16.0076    7.2021  
1   7.0165  179.0964  13.3827    3.7656  
2   5.5020   67.4941   8.2155    3.6900  
3   5.3258   53.6569   7.3251    3.3746  
4   4.0789   37.4251   6.1176    2.6891  
5   7.5704  134.6515  11.6039    4.6890  
